In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    # StructField,
    # StringType,
    # LongType,
    # IntegerType,
    # MapType,
)
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from helpers import get_table, get_bronze
from functools import reduce

print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("silver_processing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/04/27 18:43:41 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 18:43:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4e618f59-8851-4768-811b-835cf7848ef8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 425ms :: artifacts dl 29ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# Usage events silver

Grain: one row per usage event.

Processing steps:
+ Read Bronze `usage_events`.
+ Parse the JSON payload in `data`.
+ Cast event IDs, user IDs, subscription IDs, sessions, timestamps, and payload attributes.
+ Normalize event, device, platform, feature, and content fields.
+ Quarantine null keys, malformed timestamps, invalid event types, and referential-integrity failures.
+ Deduplicate at event grain.
+ Derive event date parts and event flags for downstream engagement and churn features.
+ Upsert into Delta path `./silver/usage_events`.

## Read Bronze usage_events data

In [3]:
usage_events_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/usage_events"
df_bronze_usage_events = get_bronze(usage_events_bronze, spark=spark).withColumnRenamed("created_at", "created_at_event")
df_bronze_usage_events.show(2, truncate=False)

+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------+--------------------------+-------------------------------------------------------------------+-----------------------+
|id |data                                                                                                                                                                                                                                                                                                                                                                                                                     

In [4]:
df_bronze_usage_events.printSchema()
df_bronze_usage_events.count()

root
 |-- id: long (nullable = true)
 |-- data: string (nullable = true)
 |-- created_at_event: timestamp (nullable = true)
 |-- processed: boolean (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- source_identifier: string (nullable = true)
 |-- batch_id: string (nullable = true)



113412

In [5]:
def flatten_struct(df):
    while True:
        struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
        if not struct_cols:
            break

        new_cols = []
        for field in df.schema.fields:
            field_name = field.name
            field_type = field.dataType

            if isinstance(field_type, StructType):
                for nested_field in field_type.fields:
                    new_cols.append(
                        F.col(f"{field_name}.{nested_field.name}").alias(f"{field_name}_{nested_field.name}")
                    )
            else:
                new_cols.append(F.col(field_name))

        df = df.select(*new_cols)

    return df

# infer schema
json_rdd = df_bronze_usage_events.select("data").rdd.map(lambda r: r[0])
json_schema = spark.read.json(json_rdd).schema

# parse json
df_parsed = (
    df_bronze_usage_events
    .withColumn("parsed", F.from_json(F.col("data"), json_schema))
    .select("*", "parsed.*")
    .drop("parsed")
)


# flatten recursively
df_flat = flatten_struct(df_parsed).drop("data", "id")

In [6]:
df1 = (
    df_flat
    .withColumn("created_at_event", F.col("created_at_event").cast("timestamp"))
    .withColumn("processed", F.col("processed").cast("boolean"))
    .withColumn("ingest_time", F.col("ingest_time").cast("timestamp"))
    .withColumn("content_id", F.col("content_id").cast("bigint"))
    .withColumn("created_at", F.col("created_at").cast("string"))
    .withColumn("device_type", F.col("device_type").cast("string"))
    .withColumn("event_id", F.col("event_id").cast("string"))
    .withColumn("event_properties_duration_sec", F.col("event_properties_duration_sec").cast("bigint"))
    .withColumn("event_properties_referrer", F.col("event_properties_referrer").cast("string"))
    .withColumn("event_timestamp", F.col("event_timestamp").cast("string"))
    .withColumn("event_type", F.col("event_type").cast("string"))
    .withColumn("feature_name", F.col("feature_name").cast("string"))
    .withColumn("platform", F.col("platform").cast("string"))
    .withColumn("session_id", F.col("session_id").cast("string"))
    .withColumn("subscription_id", F.col("subscription_id").cast("bigint"))
    .withColumn("user_id", F.col("user_id").cast("bigint"))
)

In [7]:
df1.show(1, truncate=False)

+--------------------------+---------+--------------------------+-------------------------------------------------------------------+-----------------------+----------+--------------------------+-----------+------------------------------------+-----------------------------+-------------------------+--------------------------+----------+------------+--------+------------------------------------+---------------+-------+
|created_at_event          |processed|ingest_time               |source_identifier                                                  |batch_id               |content_id|created_at                |device_type|event_id                            |event_properties_duration_sec|event_properties_referrer|event_timestamp           |event_type|feature_name|platform|session_id                          |subscription_id|user_id|
+--------------------------+---------+--------------------------+-------------------------------------------------------------------+-----------------------

In [8]:
df1.select("device_type").distinct().show()

[Stage 24:======================================>                   (2 + 1) / 3]

+-----------+
|device_type|
+-----------+
|     mobile|
|     tablet|
|        web|
+-----------+



## Deduplicate at event grain

In [9]:
w = Window.partitionBy("event_id","event_timestamp").orderBy(F.col("ingest_time").desc())

df_usage_ranked = df1.withColumn("rn", F.row_number().over(w))

df2 = df_usage_ranked.filter(F.col("rn") == 1).drop("rn", "validation_error")
df2_quarantine = (
    df_usage_ranked
    .filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate event_id"))
)

df2.count(), df2_quarantine.count()

(113412, 0)

## Validate referential integrity

In [10]:
silver_users_path = "./silver/users"
silver_subscriptions_path = "./silver/subscriptions"

df_silver_users = spark.read.format("delta").load(silver_users_path)
df_silver_subscriptions = spark.read.format("delta").load(silver_subscriptions_path)

df_users_ref = (
    df_silver_users
    .select(F.col("user_id").cast("bigint").alias("user_id"))
    .withColumn("_user_exists", F.lit(True))
)
df_subscriptions_ref = (
    df_silver_subscriptions
    .select(F.col("subscription_id").cast("bigint").alias("subscription_id"))
    .dropDuplicates()
    .withColumn("_subscription_exists", F.lit(True))
)

df_usage_with_refs = (
    df2.alias("ue")
    .join(df_users_ref, on="user_id", how="left")
    .withColumn("is_missing_user", F.col("_user_exists").isNull())
    .join(df_subscriptions_ref, on="subscription_id", how="left")
    .withColumn("is_missing_subscription", F.col("subscription_id").isNotNull() & F.col("_subscription_exists").isNull())
    .drop("_user_exists", "_subscription_exists")
)

df3 = (
    df_usage_with_refs
    .filter(~F.col("is_missing_user") & ~F.col("is_missing_subscription"))
    .drop("is_missing_user", "is_missing_subscription")
)

df3_quarantine = (
    df_usage_with_refs
    .filter(F.col("is_missing_user") | F.col("is_missing_subscription"))
    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(F.col("is_missing_user"), F.lit("user_id not found in silver users")),
            F.when(F.col("is_missing_subscription"), F.lit("subscription_id not found in silver subscriptions"))
        )
    )
    .drop("is_missing_user", "is_missing_subscription")
)

df3.count(), df3_quarantine.count()

(101937, 11475)

## Get all quarantine tables

In [11]:
quarantine_dfs = [
    df2_quarantine,
    df3_quarantine
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)

df_quarantine_all.count()

11475

In [12]:
df_usage_event_silver = df3

In [13]:
df_usage_event_silver.columns

['subscription_id',
 'user_id',
 'created_at_event',
 'processed',
 'ingest_time',
 'source_identifier',
 'batch_id',
 'content_id',
 'created_at',
 'device_type',
 'event_id',
 'event_properties_duration_sec',
 'event_properties_referrer',
 'event_timestamp',
 'event_type',
 'feature_name',
 'platform',
 'session_id']

## Upsert strategy

In [14]:
silver_path = "./silver/usage_events"

silver_cols = df_usage_event_silver.columns

df_upsert = df_usage_event_silver.select(*silver_cols)

w = Window.partitionBy("event_id", "event_timestamp").orderBy(F.col("ingest_time").desc())
df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(df_upsert.alias("source"), "target.event_id = source.event_id")
        .whenMatchedUpdate(set={c: f"source.{c}" for c in silver_cols if c != "event_id"})
        .whenNotMatchedInsert(values={c: f"source.{c}" for c in silver_cols})
        .execute()
    )
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table already exists


In [15]:
df_usage_events_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_usage_events_silver_read.show(5, truncate=False)
df_usage_events_silver_read.count()

[Stage 141:=======================================================(50 + 0) / 50]

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
spark.stop()